# ¿El clúster aporta valor predictivo más allá de las covariables? (R2-6)

Notebook independiente para responder al Revisor 2 (comentario 6). Compara, de forma NO circular, un clustering alternativo construido SOLO con covariables socioeconómicas/de acceso (sin los 5 módulos de puntaje) contra el clustering publicado (que SÍ usa los puntajes — por eso no sirve como prueba justa).

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/valor_predictivo/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/valor_predictivo'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Partición 'publicada' de referencia (K=8, misma metodología del manuscrito)

In [ ]:
# Partición base K=8 (misma metodología del manuscrito: UMAP fit sobre
# 80,000 filas, transform sobre el resto; K-Means K=8), para tener una
# referencia 'publicada' propia sin depender de otros notebooks.
SEED_BASE = 42
K = 8
N_FIT = 80_000
N_EVAL = 50_000

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

rng_fit = np.random.default_rng(seed=SEED_BASE)
idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
rng_eval = np.random.default_rng(seed=0)
idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)

log("Ajustando UMAP base (K=8, seed=42)...")
reducer_base = umap_cpu.UMAP(n_components=2, random_state=SEED_BASE, n_neighbors=10,
                              low_memory=True, n_jobs=-1)
reducer_base.fit(X_full[idx_fit])
emb_eval_base = reducer_base.transform(X_full[idx_eval])
km_base = MiniBatchKMeans(n_clusters=K, random_state=SEED_BASE, n_init="auto", batch_size=10_000)
labels_pub = km_base.fit_predict(emb_eval_base)
log(f"Partición base lista. Tamaños: {np.bincount(labels_pub)}")

## 4. Clustering alternativo SOLO con covariables SES/acceso (sin puntajes)

In [ ]:
y_eval = df_filtrado_full["PUNT_GLOBAL"].values[idx_eval].astype(float)
mask_y = ~np.isnan(y_eval)
print(f"PUNT_GLOBAL disponible en {mask_y.sum()}/{len(y_eval)} filas del conjunto de evaluación")

X_ses_full = np.hstack([X_full[:, :5], X_full[:, 10:]])  # excluye las 5 columnas de puntaje
print(f"X_ses_full shape: {X_ses_full.shape}")

seed_ses = 600
rng_fit = np.random.default_rng(seed=seed_ses)
idx_fit_ses = rng_fit.choice(n_total, size=N_FIT, replace=False)
reducer_ses = umap_cpu.UMAP(n_components=2, random_state=seed_ses, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
reducer_ses.fit(X_ses_full[idx_fit_ses])
emb_eval_ses = reducer_ses.transform(X_ses_full[idx_eval])
km_ses = MiniBatchKMeans(n_clusters=K, random_state=seed_ses, n_init="auto", batch_size=10_000)
labels_ses = km_ses.fit_predict(emb_eval_ses)
print(f"Clustering SES-only listo. Tamaños: {np.bincount(labels_ses)}")

## 5. Comparar R² con Ridge y HistGBM (con y sin clúster como predictor)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold

X_cov = X_ses_full[idx_eval][mask_y]
y = y_eval[mask_y]
labels_ses_m = labels_ses[mask_y]
labels_pub_m = labels_pub[mask_y]

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
dummies_ses = ohe.fit_transform(labels_ses_m.reshape(-1, 1))
dummies_pub = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit_transform(
    labels_pub_m.reshape(-1, 1))

X_A = X_cov
X_B_ses = np.hstack([X_cov, dummies_ses])
X_C_pub = np.hstack([X_cov, dummies_pub])
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Ridge: Modelo A (solo covariables)...")
r2_A = cross_val_score(Ridge(alpha=1.0), X_A, y, cv=cv, scoring="r2")
print(f"  R2 = {r2_A.mean():.4f} +- {r2_A.std():.4f}")

print("Ridge: Modelo B (covariables + clúster SES-only, NO circular)...")
r2_B = cross_val_score(Ridge(alpha=1.0), X_B_ses, y, cv=cv, scoring="r2")
print(f"  R2 = {r2_B.mean():.4f} +- {r2_B.std():.4f}")

print("Ridge: Modelo C (covariables + clúster PUBLICADO, ilustrativo/circular)...")
r2_C = cross_val_score(Ridge(alpha=1.0), X_C_pub, y, cv=cv, scoring="r2")
print(f"  R2 = {r2_C.mean():.4f} +- {r2_C.std():.4f}")

print("HistGBM: baseline (solo covariables)...")
r2_gbm_A = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_A, y, cv=cv, scoring="r2")
print(f"  R2 = {r2_gbm_A.mean():.4f} +- {r2_gbm_A.std():.4f}")

print("HistGBM: covariables + clúster SES-only...")
r2_gbm_B = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_B_ses, y, cv=cv, scoring="r2")
print(f"  R2 = {r2_gbm_B.mean():.4f} +- {r2_gbm_B.std():.4f}")

## 6. Resumen

In [ ]:
resumen = {
    'ridge_A_solo_covariables': float(r2_A.mean()),
    'ridge_B_mas_cluster_ses_no_circular': float(r2_B.mean()),
    'ridge_C_mas_cluster_publicado_CIRCULAR': float(r2_C.mean()),
    'gbm_A_solo_covariables': float(r2_gbm_A.mean()),
    'gbm_B_mas_cluster_ses_no_circular': float(r2_gbm_B.mean()),
    'delta_r2_ridge_no_circular': float(r2_B.mean() - r2_A.mean()),
    'delta_r2_gbm_no_circular': float(r2_gbm_B.mean() - r2_gbm_A.mean()),
    'delta_r2_ridge_circular_ilustrativo': float(r2_C.mean() - r2_A.mean()),
}
json.dump(resumen, open(os.path.join(OUT_DIR, 'resultados.json'), 'w'), indent=2)
print(json.dumps(resumen, indent=2))
print("\nValores esperados (manuscrito): ΔR² no-circular ≈ 0.0004-0.0007 "
      "(insignificante); ΔR² circular ilustrativo ≈ 0.069.")

## 6. Figura (Supplementary Figure S6)

In [ ]:
import matplotlib.pyplot as plt

labels = ['Solo covariables\n(Ridge, lineal)', '+ cl\u00faster SES-only\n(Ridge, no circular)',
          '+ cl\u00faster PUBLICADO\n(Ridge, CIRCULAR)', 'Solo covariables\n(HistGBM, no lineal)',
          '+ cl\u00faster SES-only\n(HistGBM, no circular)']
values = [r2_A.mean(), r2_B.mean(), r2_C.mean(), r2_gbm_A.mean(), r2_gbm_B.mean()]
colors = ['#A0A0A0', '#3B6FA0', '#B33F3F', '#A0A0A0', '#3B6FA0']

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=150)
bars = ax.bar(labels, values, color=colors, alpha=0.9)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.006, f'{v:.3f}', ha='center', fontsize=10)
ax.set_ylabel('R\u00b2 (predicci\u00f3n de PUNT_GLOBAL, validaci\u00f3n cruzada 5-fold)')
ax.set_title('\u00bfLa pertenencia al cl\u00faster mejora la predicci\u00f3n\nm\u00e1s all\u00e1 de las covariables por s\u00ed solas?')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_valor_predictivo_cluster.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S6)")
